<a href="https://colab.research.google.com/github/acastellanos-ie/NLP-MBDS-EN/blob/main/07_rag/QA_practice_with_HF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Question Answering: Extractive and Abstractive

An extractive model selects a span from a context. An abstractive model generates an answer token by token. Neither one searches for evidence unless we connect it to a retriever.

Let's inspect both approaches, when they work and what happens when the answer is missing.

In [ ]:
# @title Setup
%pip install -q "transformers==4.55.4" "sentencepiece==0.2.0" "matplotlib==3.10.5"

## Extractive QA: model and context

The model below was fine-tuned for extractive QA. We will keep the context fixed so every result is reproducible.

In [ ]:
import torch
from transformers import AutoModelForQuestionAnswering, AutoTokenizer

model_id = "distilbert/distilbert-base-cased-distilled-squad"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForQuestionAnswering.from_pretrained(model_id)
model.eval()

context = (
    "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in "
    "Paris, France. It is named after the engineer Gustave Eiffel, whose company "
    "designed and built the tower."
)

print(f"Model: {model_id}")
print(f"Context: {context}")

## Selecting an answer span

The model assigns a start logit and an end logit to every token. We search for the highest-scoring valid pair inside the context, with the end after the start and a maximum answer length of 20 tokens.

The reported score is the normalized score of the selected span among the valid spans. It is useful for comparing candidates in this example, but it is not a calibrated probability that the answer is correct.

In [ ]:
# @title Extractive QA function
def extract_answer(question, context, max_answer_length=20):
    encoded = tokenizer(
        question,
        context,
        return_tensors="pt",
        truncation="only_second",
        max_length=384,
        return_offsets_mapping=True,
    )
    sequence_ids = encoded.sequence_ids(0)
    offsets = encoded.pop("offset_mapping")[0]

    with torch.no_grad():
        output = model(**encoded)

    start_logits = output.start_logits[0]
    end_logits = output.end_logits[0]
    token_count = len(start_logits)
    positions = torch.arange(token_count)
    context_mask = torch.tensor([sequence_id == 1 for sequence_id in sequence_ids])

    span_scores = start_logits[:, None] + end_logits[None, :]
    valid_spans = (
        (positions[:, None] <= positions[None, :])
        & ((positions[None, :] - positions[:, None]) < max_answer_length)
        & context_mask[:, None]
        & context_mask[None, :]
    )
    span_scores = span_scores.masked_fill(~valid_spans, -torch.inf)

    flat_index = int(torch.argmax(span_scores))
    start_index = flat_index // token_count
    end_index = flat_index % token_count
    start_character = int(offsets[start_index, 0])
    end_character = int(offsets[end_index, 1])
    answer = context[start_character:end_character]

    valid_score_distribution = torch.softmax(span_scores[valid_spans], dim=0)
    span_score = float(valid_score_distribution.max())
    return {"answer": answer, "span_score": span_score}

## Questions answered by the context

Test three questions whose answers appear explicitly in the text.

In [ ]:
answerable_questions = [
    ("Who designed the Eiffel Tower?", "Gustave Eiffel"),
    ("Where is the Eiffel Tower?", "Champ de Mars in Paris, France"),
    ("What is the tower made of?", "wrought-iron"),
]

answerable_results = []
for question, expected in answerable_questions:
    result = extract_answer(question, context)
    answerable_results.append(result)
    verdict = "PASS" if result["answer"] == expected else "FAIL"
    print(
        f"{verdict} | {question:<35} answer={result['answer']!r:<38} "
        f"span_score={result['span_score']:.4f}"
    )

All three answers are exact spans from the context. The score changes considerably across correct answers: *Gustave Eiffel* is close to 0.99, while the longer location is around 0.39. A lower score does not automatically mean a wrong answer.

## Questions not answered by the context

Now ask for a date that is absent and for the painter of the Mona Lisa. The correct response in both cases is *I don't know*.

<div style="border: 2px solid currentColor; border-radius: 12px; padding: 28px 24px; margin: 18px 0; text-align: center;">
  <div style="font-size: 0.85em; letter-spacing: 0.08em; text-transform: uppercase; margin-bottom: 10px;">Make a prediction</div>
  <div style="font-size: 1.5em; line-height: 1.35;"><strong>If the context does not contain an answer, what will an extractive model extract?</strong></div>
</div>

In [ ]:
unanswerable_questions = [
    "In what year was the Eiffel Tower completed?",
    "Who painted the Mona Lisa?",
]

unanswerable_results = []
for question in unanswerable_questions:
    result = extract_answer(question, context)
    unanswerable_results.append(result)
    print(
        f"Q: {question}\n"
        f"A: {result['answer']!r} | span_score={result['span_score']:.4f}\n"
    )

The model returns *Gustave Eiffel* for both unsupported questions. The second wrong answer even receives a score above 0.96.

This model was trained to select an answer span. Our decoding code also forces the result to remain inside the context. Without an explicit no-answer mechanism, it will often choose the best-looking span even when no valid answer exists.

## A retrieval error becomes a QA error

Give the Eiffel Tower question an unrelated context.

In [ ]:
irrelevant_context = (
    "The Pacific Ocean is the largest and deepest ocean on Earth. "
    "It extends from the Arctic Ocean to the Southern Ocean."
)
retrieval_error = extract_answer("Who designed the Eiffel Tower?", irrelevant_context)

print(f"Context: {irrelevant_context}")
print(f"Answer: {retrieval_error['answer']!r}")
print(f"Span score: {retrieval_error['span_score']:.4f}")

In [ ]:
import matplotlib.pyplot as plt

qa_score_labels = [
    "Supported: designer",
    "Supported: location",
    "Supported: material",
    "Missing: completion year",
    "Missing: Mona Lisa painter",
    "Wrong context",
]
qa_span_scores = [
    *(result["span_score"] for result in answerable_results),
    *(result["span_score"] for result in unanswerable_results),
    retrieval_error["span_score"],
]
qa_verdicts = ["correct"] * 3 + ["wrong"] * 3
bar_colors = ["C0"] * 3 + ["C3"] * 3

fig, axis = plt.subplots(figsize=(9, 4.2))
bars = axis.barh(qa_score_labels, qa_span_scores, color=bar_colors)
axis.invert_yaxis()
axis.bar_label(
    bars,
    labels=[
        f"{score:.2f}  {verdict}"
        for score, verdict in zip(qa_span_scores, qa_verdicts)
    ],
    padding=4,
)
axis.set(
    title="Extractive span score does not measure answerability",
    xlabel="Normalized score of the selected span",
)
axis.set_xlim(0, 1.18)
axis.set_xticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
axis.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

The model extracts *The Pacific Ocean*. Its span score is low, but the central problem happened earlier: the context does not contain the evidence. A QA model cannot repair a retrieval failure.

The bars also show why a simple confidence threshold would not solve answerability. The correct location span scores only about 0.39, while one unsupported answer scores above 0.96. The score ranks spans inside the supplied context; it does not establish that the context contains an answer.

## Abstractive QA

The original notebook also moved from span extraction to T5 generation. We keep that transition, but use **google/flan-t5-small**, which was instruction-tuned and can follow an explicit grounding rule.

The prompt asks the model to use only the supplied context and to say *Not enough information* when the answer is absent. This is an instruction, not a guarantee.

In [ ]:
from transformers import AutoModelForSeq2SeqLM

generative_model_id = "google/flan-t5-small"
generative_tokenizer = AutoTokenizer.from_pretrained(generative_model_id)
generative_model = AutoModelForSeq2SeqLM.from_pretrained(generative_model_id)
generative_model.eval()

print(f"Model: {generative_model_id}")
print(f"Parameters: {sum(p.numel() for p in generative_model.parameters()):,}")

FLAN-T5 Small has about 77 million parameters. Unlike the extractive model, it has a decoder and is not constrained to copy a contiguous span. That extra freedom allows abstention, but it also allows unsupported generation.

In [ ]:
def generate_answer(question, context):
    prompt = (
        "Answer using only the context. If the answer is missing, say "
        f"Not enough information. Context: {context} "
        f"Question: {question} Answer:"
    )
    encoded = generative_tokenizer(prompt, return_tensors="pt", truncation=True)
    with torch.inference_mode():
        output_ids = generative_model.generate(
            **encoded,
            max_new_tokens=20,
            do_sample=False,
        )
    return generative_tokenizer.decode(output_ids[0], skip_special_tokens=True)


abstractive_questions = [
    question for question, _ in answerable_questions
] + unanswerable_questions
abstractive_results = [
    generate_answer(question, context)
    for question in abstractive_questions
]

for question, answer in zip(abstractive_questions, abstractive_results):
    print(f"Q: {question}\nA: {answer}\n")

For the supported questions, FLAN-T5 returns *Gustave Eiffel*, *Paris, France* and *wrought-iron*. The location is a shorter paraphrase rather than the longer extractive span.

For the two unsupported questions it returns *Not enough* or *Not enough information*. The wording is not perfectly consistent, but it does not invent a year or a painter in this run. This is better abstention than the extractive model, which was forced to choose *Gustave Eiffel* twice.

In [ ]:
abstractive_retrieval_error = generate_answer(
    "Who designed the Eiffel Tower?",
    irrelevant_context,
)
print(f"Wrong context answer: {abstractive_retrieval_error!r}")

With the Pacific Ocean context, the abstractive model says *Not enough information* while the extractive model returned *The Pacific Ocean*. Prompted abstention helps here, but retrieval is still the upstream component that decides whether useful evidence reaches either reader.

In [ ]:
# @title Consistency checks
assert [result["answer"] for result in answerable_results] == [
    "Gustave Eiffel",
    "Champ de Mars in Paris, France",
    "wrought-iron",
]
assert [result["answer"] for result in unanswerable_results] == [
    "Gustave Eiffel",
    "Gustave Eiffel",
]
assert unanswerable_results[1]["span_score"] > 0.90
assert retrieval_error["answer"] == "The Pacific Ocean"
assert abstractive_results == [
    "Gustave Eiffel",
    "Paris, France",
    "wrought-iron",
    "Not enough",
    "Not enough information",
]
assert abstractive_retrieval_error == "Not enough information"
print("Checks passed: notebook text and outputs are consistent.")

# Takeaway

- Extractive QA selects a span; it does not retrieve the context.
- Start and end scores can identify precise answers when the evidence is present.
- A high span score is not a calibrated guarantee that the question is answerable.
- Abstractive QA can paraphrase or abstain, but it can also generate text that is not supported.
- If retrieval supplies the wrong context, the QA model can return a confident but irrelevant span.
- RAG needs both retrieval evaluation and answer-grounding checks.